In [6]:
import time
import json
from typing import Dict, Any, Optional

In [7]:
class MockObservabilitySDK:
    """
    Simulates a production cloud observability client (like Langfuse or LangSmith).
    It manages tracing sessions asynchronously.
    """
    def __init__(self):
        print("[SDK] Initialized connection to Observability Cloud Endpoint.")

    def create_trace(self, name: str, user_id: str) -> Dict[str, Any]:
        """Creates a top-level execution trace container."""
        return {
            "trace_name": name,
            "user_id": user_id,
            "spans": []
        }

    def log_span(self, trace: Dict[str, Any], name: str, input_data: Any, output_data: Any, metadata: Dict[str, Any]):
        """Appends a completed operational unit (span) to the active trace."""
        span_entry = {
            "span_name": name,
            "input": input_data,
            "output": output_data,
            "metadata": metadata
        }
        trace["spans"].append(span_entry)
        print(f"[SDK Cloud Sync] → Captured Span: '{name}' shipped to dashboard.")

In [8]:
# Initialize our third-party platform client
observability_platform = MockObservabilitySDK()

def production_ai_pipeline(user_query: str, client_id: str):
    """
    A production pipeline wrapper showing how third-party tracing 
    wraps cleanly around your existing system architecture.
    """
    # 1. Start a high-level trace tracking this user session
    user_trace = observability_platform.create_trace(
        name="User Support Chatbot Run", 
        user_id=client_id
    )
    
    # --- STEP 1: Content Moderation Guardrail ---
    start_time = time.time()
    is_safe = True # Simulated safety check passing
    duration = time.time() - start_time
    
    observability_platform.log_span(
        trace=user_trace,
        name="Guardrail: Content Safety Check",
        input_data={"text": user_query},
        output_data={"is_safe": is_safe},
        metadata={"latency_ms": round(duration * 1000, 2), "engine": "internal-regex"}
    )
    
    # --- STEP 2: Actual LLM Processing ---
    start_time = time.time()
    mock_llm_reply = "Your subscription renewal date is October 14th."
    duration = time.time() - start_time
    
    observability_platform.log_span(
        trace=user_trace,
        name="LLM Generation: OpenAI gpt-4o",
        input_data={"prompt": f"Answer this query: {user_query}"},
        output_data={"choices": [{"text": mock_llm_reply}]},
        metadata={
            "latency_ms": round(duration * 1000, 2),
            "usage": {"prompt_tokens": 120, "completion_tokens": 45},
            "temperature": 0.2
        }
    )
    
    # --- STEP 3: Final Dashboard Printout ---
    print("\n" + "="*60)
    print("PRODUCTION WEB DASHBOARD VIEW (Simulated UI JSON Graph):")
    print("="*60)
    print(json.dumps(user_trace, indent=2))

[SDK] Initialized connection to Observability Cloud Endpoint.


In [9]:
# Execute the traced production payload
production_ai_pipeline(
    user_query="When does my account renew?", 
    client_id="user_89234"
)

[SDK Cloud Sync] → Captured Span: 'Guardrail: Content Safety Check' shipped to dashboard.
[SDK Cloud Sync] → Captured Span: 'LLM Generation: OpenAI gpt-4o' shipped to dashboard.

PRODUCTION WEB DASHBOARD VIEW (Simulated UI JSON Graph):
{
  "trace_name": "User Support Chatbot Run",
  "user_id": "user_89234",
  "spans": [
    {
      "span_name": "Guardrail: Content Safety Check",
      "input": {
        "text": "When does my account renew?"
      },
      "output": {
        "is_safe": true
      },
      "metadata": {
        "latency_ms": 0.0,
        "engine": "internal-regex"
      }
    },
    {
      "span_name": "LLM Generation: OpenAI gpt-4o",
      "input": {
        "prompt": "Answer this query: When does my account renew?"
      },
      "output": {
        "choices": [
          {
            "text": "Your subscription renewal date is October 14th."
          }
        ]
      },
      "metadata": {
        "latency_ms": 0.0,
        "usage": {
          "prompt_tokens": 12